Open this notebook from the anonymous repository or upload it directly to Colab. Before running the setup cell, define PROBING_VLMS_REPO_URL and PROBING_VLMS_RELEASE_BASE in the environment.

# Where does Wall physics become readable?

This notebook is a standalone Wall version of the UMaze layerwise probe walkthrough. It asks **which physical variables can be recovered with a simple linear map, from which model feature, and at which layer**.

It uses the same implementation as the UMaze notebook:

- the world-model checkpoints stay frozen;
- one deterministic four-frame window is selected from each complete trajectory;
- DINO is tested with single-frame, first-difference, and second-difference features;
- predictor layers are tested from their contextual visual features;
- probes use the same ridge value, train/validation/locked-test separation, shuffled-label test, position-only test, position-residual test, and complete-window bootstrap intervals;
- straightening OFF and ON are evaluated on exactly the same windows and splits.

Here, **readable** only means that one linear map can recover a variable on held-out data. It does not show that the planner uses that variable.

## One-click setup

This setup does not require Google Drive, R2 credentials, or manually uploaded scripts. It:

1. clones the public probing repository;
2. installs the small set of missing Colab packages;
3. reconstructs and verifies the checked-in Wall checkpoints;
4. downloads and verifies the `wall_single` dataset from the `probing-VLMs` public GitHub Release.

The first run downloads about 2 GB of data and checkpoint files. Use a GPU runtime. Rerunning the notebook in the same Colab session reuses the downloaded files and activation caches.

In [ ]:
import hashlib, os, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path

REPO_URL = os.environ.get("PROBING_VLMS_REPO_URL", "").strip()
RELEASE_BASE = os.environ.get("PROBING_VLMS_RELEASE_BASE", "").rstrip("/")
REPO_BRANCH = "initial-release"
COLAB_REPO = Path("/content/probing-VLMs")
ASSET_ROOT = Path("/content/wall_probe_assets")
DATA_ARCHIVE = ASSET_ROOT / "wall_single.zip"
DATA_URL = f"{RELEASE_BASE}/wall-probe-data-v1/wall_single.zip"
DATA_SHA256 = "2b4ae4ed0ad03b337efac637f17752e7e7e27f864fec39dc25b51fef490c980d"
DATA_EXTRACT = ASSET_ROOT / "data"

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_URL or not RELEASE_BASE:
        raise RuntimeError("Set PROBING_VLMS_REPO_URL and PROBING_VLMS_RELEASE_BASE to the anonymous repository before running in Colab.")
    if not (COLAB_REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--branch", REPO_BRANCH, "--single-branch",
            REPO_URL, str(COLAB_REPO),
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(COLAB_REPO), "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(COLAB_REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)

    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "decord", "einops", "omegaconf", "hydra-core==1.3.2",
        "pymunk", "gym==0.23.1",
    ], check=True)

os.chdir(COLAB_REPO if COLAB_REPO.exists() else Path.cwd())
ASSET_ROOT.mkdir(parents=True, exist_ok=True)

checkpoint_root = Path("artifacts/checkpoints/wall_dino_projector_full_final")
off_checkpoint = checkpoint_root / "off/model_20.pth"
on_checkpoint = checkpoint_root / "on/model_20.pth"
if not off_checkpoint.exists() or not on_checkpoint.exists():
    subprocess.run(["bash", str(checkpoint_root / "restore_checkpoints.sh")], check=True)

if not DATA_ARCHIVE.exists():
    temporary = DATA_ARCHIVE.with_suffix(".download")
    print("Downloading the Wall dataset from the repository release...")
    urllib.request.urlretrieve(DATA_URL, temporary)
    temporary.replace(DATA_ARCHIVE)

digest = hashlib.sha256()
with DATA_ARCHIVE.open("rb") as handle:
    for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()
if actual_sha256 != DATA_SHA256:
    DATA_ARCHIVE.unlink(missing_ok=True)
    raise RuntimeError(f"Wall dataset checksum mismatch: {actual_sha256}")
print(f"Verified wall_single.zip SHA-256: {actual_sha256}")

marker = DATA_EXTRACT / ".complete"
if not marker.exists():
    shutil.rmtree(DATA_EXTRACT, ignore_errors=True)
    DATA_EXTRACT.mkdir(parents=True)
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        archive.extractall(DATA_EXTRACT)
    marker.write_text("complete")

data_candidates = [path.parent for path in DATA_EXTRACT.rglob("states.pth")]
if not data_candidates:
    raise FileNotFoundError("The Wall archive did not contain states.pth")
DATA_DIR = data_candidates[0]

OUTPUT_DIR = Path(os.environ.get("WALL_PROBE_OUTPUT", "/content/wall_probe_results"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "repo": str(Path.cwd()),
    "data": str(DATA_DIR),
    "off_checkpoint": str(off_checkpoint),
    "on_checkpoint": str(on_checkpoint),
    "output": str(OUTPUT_DIR),
})

## Experimental design

Each cached feature has shape `[trajectory window, sampled frame, feature]`. The probe receives one feature vector at a time; it never receives pixels, model weights, future labels, or maze coordinates, except in the separate position-only control.

| Model feature | Position probe | Velocity probe | Acceleration probe |
|---|---|---|---|
| DINO layer | one-frame feature | first temporal difference | second temporal difference |
| Predictor layer | contextual feature | contextual feature | contextual feature |

Every probe is fitted separately for one condition, layer, feature type, target, and evaluation split. Feature scaling is learned from the probe-training set only. Ridge regression uses λ = 10, matching the UMaze notebook.

In [ ]:
import csv, gc, json, math, random, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from datasets.img_transforms import default_transform
from datasets.wall_dset import WallDataset
from scripts.umaze_probe_walkthrough import (
    align_representation, bootstrap_metric_ci, build_motion_targets,
    direction_scores, episode_group_train_val_test_split,
    fit_probe, fit_probe_grouped, group_flat_predictions_by_window,
    grouped_metric, grouped_regression_summary,
    load_activation_cache, select_then_test_representations,
    trajectory_bootstrap_metric_ci,
    mask_slow_directions, readability_onset, regression_scores,
    residualize_against_position, save_activation_cache,
    shuffled_label_score, spatial_holdout_split,
)
import scripts.activation_extraction as activation_tools

plt.style.use("seaborn-v0_8-whitegrid")

SEED = 0
RIDGE = 10.0
NUM_FRAMES = 4
FRAME_SKIP = 5
STEP_DT = 1.0
BATCH_SIZE = 8
BOOTSTRAP_REPEATS = 1000
SHUFFLE_REPEATS = 20
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def write_rows(path, rows):
    if not rows:
        return
    fields = list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader(); writer.writerows(rows)

def show_rows(rows, columns=None, limit=20):
    rows = list(rows)[:limit]
    if not rows:
        print("(no rows)"); return
    columns = columns or list(rows[0])
    widths = {
        key: min(38, max(len(key), *(len(f"{row.get(key, '')}") for row in rows)))
        for key in columns
    }
    print(" | ".join(key.ljust(widths[key]) for key in columns))
    print("-+-".join("-" * widths[key] for key in columns))
    for row in rows:
        print(" | ".join(f"{row.get(key, '')}"[:widths[key]].ljust(widths[key]) for key in columns))

def grouped(rows, keys):
    result = {}
    for row in rows:
        result.setdefault(tuple(row[key] for key in keys), []).append(row)
    return result

# Keep the UMaze zero-padding implementation, but print each mismatch only once.
_padding_warnings = set()
def encode_stream_once(module, values, name):
    expected = int(module.patch_embed.in_channels)
    actual = int(values.shape[-1])
    if expected != actual:
        if actual > expected:
            raise ValueError(f"{name} has {actual} channels but checkpoint expects {expected}")
        key = (name, actual, expected)
        if key not in _padding_warnings:
            print(f"Padding {name} from {actual} to {expected} channels for the legacy checkpoint")
            _padding_warnings.add(key)
        values = F.pad(values, (0, expected - actual))
    return module(values)

activation_tools.encode_stream = encode_stream_once
print({"device": str(DEVICE), "ridge": RIDGE, "bootstrap_repeats": BOOTSTRAP_REPEATS})

## 1. Load every Wall trajectory and choose matched windows

The OSF release contains 1,920 complete Wall trajectories. We select one valid four-frame window from every trajectory with a fixed seed. This gives broad trajectory coverage without allowing one long episode to dominate the probe dataset.

The same 1,920 `(episode, start frame)` choices are used for straightening OFF and ON. The cell also verifies that no trajectory is repeated.

In [ ]:
dataset = WallDataset(
    data_path=str(DATA_DIR), transform=default_transform(224), normalize_action=True
)
dataset.seq_lengths = torch.full((len(dataset),), int(dataset.traj_len), dtype=torch.long)

choices = []
for episode in range(len(dataset)):
    max_start = int(dataset.traj_len) - 1 - FRAME_SKIP * (NUM_FRAMES - 1)
    if max_start < 0:
        continue
    episode_rng = random.Random(SEED * 1_000_003 + episode)
    choices.append((episode, episode_rng.randint(0, max_start)))

choices = np.asarray(choices, dtype=np.int64)
assert len(choices) == len(dataset) == 1920
assert len(np.unique(choices[:, 0])) == len(choices)
np.save(OUTPUT_DIR / "matched_window_choices.npy", choices)

print({
    "complete_trajectories": len(dataset),
    "unique_windows": len(choices),
    "frames_per_window": NUM_FRAMES,
    "environment_steps_between_frames": FRAME_SKIP,
})

## 2. Collect intermediate features from both checkpoints

The checkpoint is frozen during this step. For each DINO block, we store the class token, the mean patch feature, and the learned projected aggregate. For each predictor block, we store the mean contextual visual feature.

The OFF and ON caches use the same images, states, actions, and window choices. If a complete cache already exists in the current runtime, it is reused. Delete its folder after changing a checkpoint or sampling setting.

In [ ]:
CHECKPOINTS = {"off": off_checkpoint, "on": on_checkpoint}
CACHE_DIRS = {condition: OUTPUT_DIR / f"activation_cache_{condition}" for condition in CHECKPOINTS}

def cache_paths(directory):
    return directory / "cache.npz", directory / "metadata.json"

def load_or_collect(condition):
    cache_path, metadata_path = cache_paths(CACHE_DIRS[condition])
    if cache_path.exists() and metadata_path.exists():
        reps, states, actions, cached_choices, metadata = load_activation_cache(cache_path)
        if np.array_equal(cached_choices, choices):
            print(f"Loaded matched {condition.upper()} cache")
            return reps, states, actions, metadata
        print(f"Ignoring stale {condition.upper()} cache because its windows differ")

    CACHE_DIRS[condition].mkdir(parents=True, exist_ok=True)
    modules = activation_tools.load_checkpoint(CHECKPOINTS[condition], DEVICE)
    reps, states, actions = activation_tools.collect_activations(
        modules, dataset, choices.tolist(), BATCH_SIZE, FRAME_SKIP, NUM_FRAMES, DEVICE,
        include_kinds={"cls", "pooled_patches", "projected_aggregate", "pooled_visual"},
    )
    metadata = {
        "condition": condition,
        "checkpoint": str(CHECKPOINTS[condition]),
        "seed": SEED,
        "num_windows": len(choices),
        "num_frames": NUM_FRAMES,
        "frameskip": FRAME_SKIP,
        "ridge": RIDGE,
    }
    save_activation_cache(cache_path, reps, states, actions, choices, metadata)
    metadata_path.write_text(json.dumps(metadata, indent=2))
    del modules
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return reps, states, actions, metadata

representations = {}
shared_states = shared_actions = None
cache_metadata = {}
for condition in ("off", "on"):
    reps, states, actions, metadata = load_or_collect(condition)
    representations[condition] = reps
    cache_metadata[condition] = metadata
    if shared_states is None:
        shared_states, shared_actions = states, actions
    else:
        np.testing.assert_allclose(states, shared_states)
        np.testing.assert_allclose(actions, shared_actions)

inventory = [
    {"condition": condition, "representation": name, "shape": str(value.shape)}
    for condition, reps in representations.items() for name, value in sorted(reps.items())
]
show_rows(inventory, limit=12)
print("Representations per condition:", {key: len(value) for key, value in representations.items()})

## 3. Build physical targets and inspect the data

Position, velocity, acceleration, speed, and direction are constructed exactly as in the UMaze notebook. The plots below show where the agent moves quickly or slowly and whether motion is strongly tied to location. This matters because a probe can otherwise appear to read motion by recognizing a place where the agent usually moves in a particular way.

In [ ]:
targets = build_motion_targets(shared_states, FRAME_SKIP, STEP_DT)
position = targets["position"].reshape(-1, 2)
speed = targets["speed"].reshape(-1)
acceleration_magnitude = targets["acceleration_magnitude"].reshape(-1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
points = axes[0].scatter(position[:, 0], position[:, 1], c=speed, s=7, cmap="viridis")
axes[0].set(
    title="Where Wall trajectories move quickly or slowly",
    xlabel="Agent x position (environment units)",
    ylabel="Agent y position (environment units)",
    aspect="equal",
)
fig.colorbar(points, ax=axes[0], label="Speed (position units per environment step)")
axes[1].hist(speed, bins=40)
axes[1].set(title="Velocity-target magnitude", xlabel="Speed", ylabel="Frame-level samples")
axes[2].hist(acceleration_magnitude[np.isfinite(acceleration_magnitude)], bins=40)
axes[2].set(title="Acceleration-target magnitude", xlabel="Acceleration magnitude", ylabel="Frame-level samples")
fig.savefig(OUTPUT_DIR / "dataset_motion_overview.png", dpi=180)
plt.show()

weighted, x_edges, y_edges = np.histogram2d(position[:, 0], position[:, 1], bins=12, weights=speed)
counts, _, _ = np.histogram2d(position[:, 0], position[:, 1], bins=[x_edges, y_edges])
speed_map = np.divide(weighted, counts, out=np.full_like(weighted, np.nan), where=counts > 0)
plt.figure(figsize=(7, 6))
plt.imshow(speed_map.T, origin="lower", cmap="viridis", aspect="equal",
           extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]])
plt.colorbar(label="Mean speed")
plt.xlabel("Agent x position")
plt.ylabel("Agent y position")
plt.title("Mean speed by location (shortcut check)")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "speed_by_position.png", dpi=180); plt.show()

## 4. Create train/validation/test splits without trajectory leakage

We use three tests:

- **Unseen trajectories:** complete trajectories are assigned to training, validation, or a locked test partition.
- **Unseen region:** the upper part of the development data is used only for validation, with a buffer removed from training.
- **Unseen doorway area:** development windows closest to their episode's doorway are used only for validation, again with a buffer.

Exploratory curves use training and validation only. The locked test trajectories are evaluated once after selecting a representation from validation results.

In [ ]:
episode_train, episode_validation, episode_test = episode_group_train_val_test_split(
    choices, validation_fraction=0.2, test_fraction=0.2, seed=SEED
)
split_counts = {"train": len(episode_train), "validation": len(episode_validation), "test": len(episode_test)}
assert split_counts == {"train": 1152, "validation": 384, "test": 384}, split_counts
print({"protocol": "trajectory_grouped_60_20_20", **split_counts})
development_idx = np.sort(np.concatenate([episode_train, episode_validation]))
anchor_position = targets["position"][:, 0]
spatial_train_local, spatial_validation_local, spatial_config = spatial_holdout_split(
    anchor_position[development_idx], axis=1, quantile=0.8, high=True, buffer_fraction=0.05
)
spatial_train = development_idx[spatial_train_local]
spatial_validation = development_idx[spatial_validation_local]

door_points = []
for episode, start in choices:
    door = np.asarray(dataset.door_locations[int(episode), int(start)]).reshape(-1)
    if door.size >= 2:
        door_xy = door[:2].astype(float)
    else:
        wall = np.asarray(dataset.wall_locations[int(episode), int(start)]).reshape(-1)
        door_xy = np.array([float(wall[0]), float(door[0])])
    door_points.append(door_xy)
door_points = np.asarray(door_points)
door_distance = np.linalg.norm(anchor_position - door_points, axis=1)
door_threshold = float(np.quantile(door_distance[development_idx], 0.20))
door_buffer = 0.05 * float(door_distance.max() - door_distance.min())
doorway_validation = development_idx[door_distance[development_idx] <= door_threshold]
doorway_train = development_idx[door_distance[development_idx] >= door_threshold + door_buffer]

splits = {
    "episode_validation": (episode_train, episode_validation),
    "spatial_validation": (spatial_train, spatial_validation),
    "doorway_validation": (doorway_train, doorway_validation),
}
print({name: {"probe_train": len(train), "validation": len(validation)} for name, (train, validation) in splits.items()})

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
axes[0].scatter(anchor_position[spatial_train, 0], anchor_position[spatial_train, 1], s=12, label="Probe training")
axes[0].scatter(anchor_position[spatial_validation, 0], anchor_position[spatial_validation, 1], s=12, label="Unseen validation region")
axes[0].set(title="Development-only spatial validation", xlabel="Window-start x", ylabel="Window-start y", aspect="equal")
axes[0].legend()
axes[1].scatter(anchor_position[doorway_train, 0], anchor_position[doorway_train, 1], s=12, label="Probe training")
axes[1].scatter(anchor_position[doorway_validation, 0], anchor_position[doorway_validation, 1], s=12, label="Near-doorway validation")
axes[1].set(title="Development-only doorway validation", xlabel="Window-start x", ylabel="Window-start y", aspect="equal")
axes[1].legend()
fig.savefig(OUTPUT_DIR / "holdout_regions.png", dpi=180)
plt.show()

## 5. Fit every layer with the same controls as UMaze

For each probe, feature scaling and ridge weights are fitted on training trajectories only. We then report held-out R², RMSE, MAE, and a bootstrap interval.

Three controls test whether an apparent motion result is misleading:

1. shuffled labels estimate chance performance;
2. a position-only model tests whether location predicts the target;
3. a residual probe tests what remains after subtracting the part predicted from position.

A convincing motion result should be above the shuffled-label threshold, outperform the position-only model, remain positive after the position component is removed, and transfer to at least one held-out location test.

In [ ]:
def evaluate_representation(condition, name, rep, variable, mode, split_name, train_idx, test_idx):
    features, labels, position_context = align_representation(rep, targets, variable, mode)
    truth, prediction, _ = fit_probe(features, labels, train_idx, test_idx, RIDGE)
    scores = regression_scores(truth, prediction)
    truth_groups, prediction_groups = group_flat_predictions_by_window(
        features, labels, test_idx, truth, prediction
    )
    ci = trajectory_bootstrap_metric_ci(
        truth_groups, prediction_groups, "r2", BOOTSTRAP_REPEATS, SEED
    )
    shuffled = shuffled_label_score(
        features, labels, train_idx, test_idx, RIDGE, SHUFFLE_REPEATS, SEED
    )
    pos_truth, pos_prediction, _ = fit_probe(
        position_context, labels, train_idx, test_idx, RIDGE
    )
    residual_labels = residualize_against_position(labels, position_context, train_idx, RIDGE)
    residual_truth, residual_prediction, _ = fit_probe(
        features, residual_labels, train_idx, test_idx, RIDGE
    )
    family, layer, kind = name.split("/", 2)
    return {
        "condition": condition,
        "representation": name,
        "family": family,
        "layer": int(layer),
        "kind": kind,
        "variable": variable,
        "mode": mode,
        "split": split_name,
        **scores,
        "ci_low": ci[0],
        "ci_high": ci[1],
        "shuffled_q95": float(np.quantile(shuffled, 0.95)),
        "position_only_r2": regression_scores(pos_truth, pos_prediction)["r2"],
        "position_residual_r2": regression_scores(residual_truth, residual_prediction)["r2"],
    }

def evaluate_condition(condition, condition_representations):
    rows = []
    for split_name, (train_idx, test_idx) in splits.items():
        for name, rep in sorted(condition_representations.items()):
            family = name.split("/", 1)[0]
            specs = [("position", "frame")]
            if family == "dino":
                specs += [("velocity", "frame"), ("velocity", "delta")]
                if rep.shape[1] >= 3:
                    specs += [("acceleration", "frame"), ("acceleration", "second_delta")]
            else:
                specs += [("velocity", "frame"), ("acceleration", "frame")]
            for variable, mode in specs:
                rows.append(evaluate_representation(
                    condition, name, rep, variable, mode, split_name, train_idx, test_idx
                ))
    return rows

metrics = []
for condition in ("off", "on"):
    print(f"Fitting {condition.upper()} probes...")
    condition_rows = evaluate_condition(condition, representations[condition])
    metrics.extend(condition_rows)
    write_rows(OUTPUT_DIR / f"{condition}_layerwise_cartesian_metrics.csv", condition_rows)

show_rows(metrics, [
    "condition", "split", "variable", "mode", "representation", "r2",
    "position_only_r2", "position_residual_r2", "shuffled_q95",
], limit=12)

## 6. Readability across DINO and predictor layers

The horizontal axis is the block index. The vertical axis is held-out R²: 1 is perfect prediction, 0 matches predicting the test-set mean, and a negative value is worse than that baseline.

DINO and predictor plots are separate because they have different depths and roles. Within each plot, compare curves for the same target rather than comparing the absolute difficulty of different targets.

In [ ]:
def primary_mode(family, variable):
    if family == "predictor" or variable == "position":
        return "frame"
    return {"velocity": "delta", "acceleration": "second_delta"}[variable]

COLORS = {"cls": "#4C78A8", "pooled_patches": "#59A14F", "projected_aggregate": "#E15759", "pooled_visual": "#B279A2"}

def plot_family(condition, family, split_name, filename):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)
    for ax, variable in zip(axes, ("position", "velocity", "acceleration")):
        candidates = [
            row for row in metrics
            if row["condition"] == condition and row["family"] == family
            and row["split"] == split_name and row["variable"] == variable
            and row["mode"] == primary_mode(family, variable)
        ]
        for (kind,), group in grouped(candidates, ["kind"]).items():
            group = sorted(group, key=lambda row: row["layer"])
            layers = np.asarray([row["layer"] for row in group])
            ax.plot(layers, [row["r2"] for row in group], marker="o", color=COLORS[kind], label=kind)
            ax.fill_between(layers, [row["ci_low"] for row in group], [row["ci_high"] for row in group], color=COLORS[kind], alpha=0.12)
        ax.axhline(0, color="black", lw=1)
        ax.set(
            title=f"Linear readability of 2-D {variable}",
            xlabel=f"{family.title()} transformer block index ℓ (0 = earliest cached block)",
            ylabel="Held-out coefficient of determination R²",
        )
        ax.legend(fontsize=7)
    split_title = {
        "episode_validation": "validation episodes",
        "spatial_validation": "development-only unseen Wall region",
        "doorway_validation": "development-only near-doorway region",
    }[split_name]
    fig.suptitle(f"{family.title()} representations on {split_title} ({condition.upper()})")
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    plt.show()

for split_name in splits:
    plot_family("off", "dino", split_name, f"off_dino_{split_name}.png")
    plot_family("off", "predictor", split_name, f"off_predictor_{split_name}.png")

## 7. Check whether DINO motion requires temporal input

The single-frame curves test whether location or appearance alone predicts motion. The temporal curves use the designated velocity and acceleration inputs from the experimental design.

If a single-frame score disappears in the location holdouts, it is likely using an environmental shortcut. A temporal score that remains above the controls in unseen trajectories and unseen locations is stronger evidence that the feature trajectory tracks physical motion.

In [ ]:
off_motion = [
    row for row in metrics
    if row["condition"] == "off" and row["family"] == "dino"
    and row["variable"] in ("velocity", "acceleration")
]
fig, axes = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)
for column, split_name in enumerate(("episode_validation", "spatial_validation", "doorway_validation")):
    for row_index, variable in enumerate(("velocity", "acceleration")):
        ax = axes[row_index, column]
        candidates = [row for row in off_motion if row["split"] == split_name and row["variable"] == variable]
        for (mode, kind), group in grouped(candidates, ["mode", "kind"]).items():
            group = sorted(group, key=lambda row: row["layer"])
            ax.plot([row["layer"] for row in group], [row["r2"] for row in group], marker="o", label=f"{mode}: {kind}")
        ax.axhline(0, color="black", lw=1)
        split_title = {
            "episode_validation": "Validation episodes",
            "spatial_validation": "Development-only unseen Wall region",
            "doorway_validation": "Development-only near-doorway region",
        }[split_name]
        ax.set(
            title=f"{split_title}: 2-D {variable}",
            xlabel="DINO transformer block index ℓ",
            ylabel="Held-out coefficient of determination R²",
        )
        ax.legend(fontsize=6)
fig.savefig(OUTPUT_DIR / "static_vs_temporal_dino.png", dpi=180)
plt.show()

control_rows = sorted(
    [row for row in off_motion if row["mode"] in ("delta", "second_delta")],
    key=lambda row: row["r2"], reverse=True,
)
show_rows(control_rows, [
    "split", "variable", "representation", "r2",
    "position_only_r2", "position_residual_r2", "shuffled_q95",
], limit=24)

## 8. Compare Cartesian and polar descriptions of motion

The Cartesian probes predict x/y components. The polar probes separate the same motion into magnitude and direction. Direction is represented by its sine and cosine so that angles near −π and π remain close.

Direction scores use mean cosine similarity: 1 means aligned, 0 means unrelated on average, and −1 means opposite. The slowest 10% of samples are excluded from direction evaluation because direction is unstable when movement is nearly zero. DINO and predictor results are shown in separate figures because their layer indices refer to different model components.

In [ ]:
def evaluate_polar(condition, condition_representations):
    rows = []
    train_idx, test_idx = splits["episode_validation"]
    for name, rep in sorted(condition_representations.items()):
        family, layer, kind = name.split("/", 2)
        velocity_mode = "delta" if family == "dino" else "frame"
        acceleration_mode = "second_delta" if family == "dino" else "frame"
        for variable, mode in [
            ("speed", velocity_mode), ("heading", velocity_mode),
            ("acceleration_magnitude", acceleration_mode),
            ("acceleration_direction", acceleration_mode),
        ]:
            features, labels, _ = align_representation(rep, targets, variable, mode)
            if variable in ("heading", "acceleration_direction"):
                magnitude_variable = "speed" if variable == "heading" else "acceleration_magnitude"
                _, magnitude, _ = align_representation(rep, targets, magnitude_variable, mode)
                labels, cutoff = mask_slow_directions(labels, magnitude, train_idx, 0.1)
            truth, prediction, _ = fit_probe(features, labels, train_idx, test_idx, RIDGE)
            is_direction = variable in ("heading", "acceleration_direction")
            score = direction_scores(truth, prediction) if is_direction else regression_scores(truth, prediction)
            truth_groups, prediction_groups = group_flat_predictions_by_window(
                features, labels, test_idx, truth, prediction
            )
            ci = trajectory_bootstrap_metric_ci(
                truth_groups, prediction_groups, "cosine" if is_direction else "r2",
                BOOTSTRAP_REPEATS, SEED,
            )
            rows.append({
                "condition": condition, "representation": name, "family": family,
                "layer": int(layer), "kind": kind, "variable": variable,
                "mode": mode, **score, "ci_low": ci[0], "ci_high": ci[1],
            })
    return rows

polar_metrics = []
for condition in ("off", "on"):
    polar_metrics.extend(evaluate_polar(condition, representations[condition]))
write_rows(OUTPUT_DIR / "matched_layerwise_polar_metrics.csv", polar_metrics)

def plot_polar_family(family, filename):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
    for ax, (variable, score_key) in zip(axes.flat, [
        ("speed", "r2"), ("heading", "cosine"),
        ("acceleration_magnitude", "r2"), ("acceleration_direction", "cosine"),
    ]):
        subset = [
            row for row in polar_metrics
            if row["condition"] == "off" and row["family"] == family
            and row["variable"] == variable
        ]
        for (kind,), group in grouped(subset, ["kind"]).items():
            group = sorted(group, key=lambda row: row["layer"])
            ax.plot(
                [row["layer"] for row in group], [row[score_key] for row in group],
                marker="o", color=COLORS[kind], label=kind,
            )
        ax.axhline(0, color="black", lw=1)
        metric_label = (
            "Held-out coefficient of determination R²"
            if score_key == "r2" else "Mean direction cosine similarity"
        )
        ax.set(
            title=variable.replace("_", " ").title(),
            xlabel=f"{family.title()} transformer block index ℓ",
            ylabel=metric_label,
        )
        ax.legend(fontsize=7)
    fig.suptitle(f"{family.title()} polar motion readability")
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    plt.show()

plot_polar_family("dino", "cartesian_vs_polar_dino_by_layer.png")
plot_polar_family("predictor", "cartesian_vs_polar_predictor_by_layer.png")
legacy_polar_plot = OUTPUT_DIR / "cartesian_vs_polar_by_layer.png"
if legacy_polar_plot.exists():
    legacy_polar_plot.unlink()

## 9. Compare straightening OFF and ON

Dashed lines show the model trained without straightening; solid lines show the straightened model. Lines with the same color use the same feature type. Both conditions use identical trajectory windows, targets, splits, and probe settings.

A positive ON-minus-OFF value means the variable became easier to read at that layer. DINO and predictor comparisons are kept in separate figures, with validation episodes, unseen Wall regions, and near-doorway regions arranged consistently across columns. This remains an observational comparison: it does not show that any readability difference caused a planning difference.

In [ ]:
STYLES = {"off": "--", "on": "-"}

def plot_matched_family(family, filename):
    split_order = ("episode_validation", "spatial_validation", "doorway_validation")
    split_titles = {
        "episode_validation": "Validation episodes",
        "spatial_validation": "Development-only unseen Wall region",
        "doorway_validation": "Development-only near-doorway region",
    }
    fig, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True)
    for column, split_name in enumerate(split_order):
        for row_index, variable in enumerate(("velocity", "acceleration")):
            ax = axes[row_index, column]
            candidates = [
                row for row in metrics
                if row["family"] == family and row["split"] == split_name
                and row["variable"] == variable
            ]
            for condition in ("off", "on"):
                condition_rows = [row for row in candidates if row["condition"] == condition]
                for (kind,), group in grouped(condition_rows, ["kind"]).items():
                    group = sorted([
                        row for row in group
                        if row["mode"] == primary_mode(family, variable)
                    ], key=lambda row: row["layer"])
                    if not group:
                        continue
                    layer = np.asarray([row["layer"] for row in group])
                    ax.plot(
                        layer, [row["r2"] for row in group], marker="o",
                        color=COLORS[kind], linestyle=STYLES[condition],
                        label=f"{condition.upper()} | {kind}",
                    )
                    ax.fill_between(
                        layer, [row["ci_low"] for row in group],
                        [row["ci_high"] for row in group],
                        color=COLORS[kind], alpha=0.06,
                    )
            ax.axhline(0, color="black", lw=1)
            ax.set(
                title=f"{split_titles[split_name]}: 2-D {variable}",
                xlabel=f"{family.title()} transformer block index ℓ",
                ylabel="Held-out coefficient of determination R²",
            )
            ax.legend(fontsize=7, ncol=2)
    fig.suptitle(f"Straightening OFF versus ON: {family.title()}")
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    plt.show()

plot_matched_family("dino", "straightening_off_vs_on_dino_by_layer.png")
plot_matched_family("predictor", "straightening_off_vs_on_predictor_by_layer.png")
for legacy_split in splits:
    for legacy_family in ("dino", "predictor"):
        legacy_plot = OUTPUT_DIR / f"matched_{legacy_family}_{legacy_split}.png"
        if legacy_plot.exists():
            legacy_plot.unlink()

off_lookup = {
    (row["representation"], row["variable"], row["mode"], row["split"]): row
    for row in metrics if row["condition"] == "off"
}
paired_deltas = []
for on_row in [row for row in metrics if row["condition"] == "on"]:
    key = (on_row["representation"], on_row["variable"], on_row["mode"], on_row["split"])
    off_row = off_lookup[key]
    paired_deltas.append({
        "split": on_row["split"], "family": on_row["family"],
        "variable": on_row["variable"], "mode": on_row["mode"],
        "representation": on_row["representation"],
        "off_r2": off_row["r2"], "on_r2": on_row["r2"],
        "on_minus_off_r2": on_row["r2"] - off_row["r2"],
    })
write_rows(OUTPUT_DIR / "straightening_on_minus_off.csv", paired_deltas)
show_rows(sorted(paired_deltas, key=lambda row: abs(row["on_minus_off_r2"]), reverse=True), limit=30)

## 10. Summarize where variables become readable

The onset is the first of two consecutive layers that exceed the shuffled-label threshold and reach at least half of the best score for that feature type. This gives a consistent summary of a curve, not a sharp boundary where information suddenly appears.

The final cell writes all tables, configuration details, and a compact summary, then packages the results into one zip file.

In [ ]:
onsets = []
for condition in ("off", "on"):
    primary_rows = [
        row for row in metrics
        if row["condition"] == condition and row["split"] == "episode_validation"
        and row["mode"] == primary_mode(row["family"], row["variable"])
    ]
    for (family, kind, variable), group in grouped(primary_rows, ["family", "kind", "variable"]).items():
        best = max(group, key=lambda row: row["r2"])
        onsets.append({
            "condition": condition, "family": family, "kind": kind,
            "variable": variable,
            "onset_layer": readability_onset(group, "r2", "shuffled_q95", 2, 0.5),
            "peak_r2": best["r2"], "peak_layer": int(best["layer"]),
        })

write_rows(OUTPUT_DIR / "readability_onsets.csv", onsets)
write_rows(OUTPUT_DIR / "matched_layerwise_cartesian_metrics.csv", metrics)
show_rows(sorted(onsets, key=lambda row: (row["condition"], row["variable"], row["family"], row["kind"])), limit=40)

summary = {
    "status": "complete",
    "environment": "wall",
    "unique_trajectories": len(choices),
    "config": {
        "seed": SEED, "ridge": RIDGE, "num_frames": NUM_FRAMES,
        "frameskip": FRAME_SKIP, "bootstrap_repeats": BOOTSTRAP_REPEATS,
        "shuffle_repeats": SHUFFLE_REPEATS,
        "splits": {name: {"train": len(train), "validation": len(test)} for name, (train, test) in splits.items()},
    },
    "checkpoint_metadata": cache_metadata,
    "onsets": onsets,
    "largest_absolute_straightening_deltas": sorted(
        paired_deltas, key=lambda row: abs(row["on_minus_off_r2"]), reverse=True
    )[:40],
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))

archive = shutil.make_archive("/content/wall_probe_results", "zip", OUTPUT_DIR)
print("WALL_PROBE_RUN_COMPLETE")
print({"output_directory": str(OUTPUT_DIR), "results_zip": archive})
print("To download in Colab, run: from google.colab import files; files.download(archive)")

## How to interpret the outputs

Read the results in this order:

1. Confirm that position is readable. This checks that the activation and target pipeline is aligned.
2. Compare the DINO single-frame and temporal motion curves. Prefer the designated temporal input for velocity and acceleration.
3. Check the position-only, residual, and shuffled-label columns before treating a motion score as meaningful.
4. Check whether the same result survives the unseen-region or doorway test.
5. Compare OFF and ON lines of the same color to isolate the straightening condition.
6. Treat the onset table as a compact description of the full curves, not as proof of a discrete emergence point.

The strongest supported statement is: a given physical variable is linearly readable from a named feature at a named layer, exceeds the controls, and transfers to a specified held-out split. Whether the planner uses that information requires a separate intervention experiment.


## Confirmatory trajectory-level evaluation

The preceding curves use development data only. For each target and model family, one shared layer, readout, and temporal construction is chosen by mean OFF/ON validation $R^2$. The same choice is fixed for both conditions before each probe is refit on training plus validation trajectories and scored once on the locked test trajectories.

Headline $R^2$, RMSE, MAE, and straightening differences include 95% percentile intervals from 1,000 complete-trajectory-window bootstrap resamples. The current checkpoint bundle contains model-training seed 0 only; independently trained seeds should be added as separate checkpoint and cache bundles.

In [ ]:
HEADLINE_BOOTSTRAP_REPEATS = 1000
AVAILABLE_MODEL_SEEDS = [0]

def wall_headline_specs(family, rep):
    specs = [("position", "frame")]
    if family == "dino":
        specs += [("velocity", "frame"), ("velocity", "delta")]
        if rep.shape[1] >= 3:
            specs += [("acceleration", "frame"), ("acceleration", "second_delta")]
    else:
        specs += [("velocity", "frame"), ("acceleration", "frame")]
    return specs

validation_selection, headline_metrics, headline_deltas = select_then_test_representations(
    representations, targets, align_representation, wall_headline_specs,
    episode_train, episode_validation, episode_test, ridge=RIDGE,
    bootstrap_repeats=HEADLINE_BOOTSTRAP_REPEATS, seed=SEED, model_seed=0,
)
write_rows(OUTPUT_DIR / "validation_selection_scores.csv", validation_selection)
write_rows(OUTPUT_DIR / "headline_selected_test_metrics.csv", headline_metrics)
write_rows(OUTPUT_DIR / "headline_straightening_deltas.csv", headline_deltas)
headline_protocol = {
    "model_training_seeds": AVAILABLE_MODEL_SEEDS,
    "split_windows": {"train": len(episode_train), "validation": len(episode_validation), "test": len(episode_test)},
    "selection": "shared candidate maximizing mean OFF/ON validation R2 within family and target",
    "test_policy": "selected representation evaluated once after refitting on train+validation",
    "bootstrap": {"unit": "complete trajectory window", "repeats": HEADLINE_BOOTSTRAP_REPEATS, "interval": "95% percentile"},
}
(OUTPUT_DIR / "headline_protocol.json").write_text(json.dumps(headline_protocol, indent=2))
show_rows(headline_metrics, ["condition", "family", "variable", "representation", "mode", "r2", "r2_ci_low", "r2_ci_high", "rmse", "rmse_ci_low", "rmse_ci_high", "mae", "mae_ci_low", "mae_ci_high"], limit=20)
show_rows(headline_deltas, limit=20)

archive = shutil.make_archive("/content/wall_probe_results_confirmatory", "zip", OUTPUT_DIR)
print({"confirmatory_results_zip": archive})
